# torch_fidelity

torch-fidelity 0.4.0 computes the standard quality scores of generative image models: Frechet Inception Distance (FID),
Kernel Inception Distance (KID), Inception Score (IS) and precision and recall, faithful to the original reference code.
torchmetrics' FID, KID and IS use its Inception network. These cells check the metric maths on inputs with known answers
and run the Inception network and the full pipeline from a weights file on disk, because the cells must not download
anything (the real weights are a 91 MB download from a GitHub release; the package fetches them on first use).

In [ ]:
# Example 1: FID maths with a known answer: same statistics give 0; shifted means give the squared distance.
import numpy as np
from torch_fidelity.metric_fid import fid_features_to_statistics, fid_statistics_to_metric
import torch

rng = np.random.default_rng(0)
a = torch.from_numpy(rng.normal(size=(2000, 8)))
b = a + 2.0  # the same data shifted by 2 in each of 8 dimensions: FID = 8 * 2**2 = 32
s1, s2 = fid_features_to_statistics(a), fid_features_to_statistics(b)
same = fid_statistics_to_metric(s1, s1, verbose=False)["frechet_inception_distance"]
shifted = fid_statistics_to_metric(s1, s2, verbose=False)["frechet_inception_distance"]
print("same", same, "shifted", shifted)
assert abs(same) < 1e-6
assert abs(shifted - 32.0) < 1e-6

In [ ]:
# Example 2: KID: two samples of one distribution score near 0, a shifted sample scores far above it.
import torch
from torch_fidelity.metric_kid import kid_features_to_metric

torch.manual_seed(0)
kw = dict(kid_subsets=5, kid_subset_size=64, kid_degree=3, kid_gamma=None, kid_coef0=1, rng_seed=2020, verbose=False)
a, b = torch.randn(512, 16), torch.randn(512, 16)
near = kid_features_to_metric(a, b, **kw)["kernel_inception_distance_mean"]
far = kid_features_to_metric(a, b + 3.0, **kw)["kernel_inception_distance_mean"]
print("near", near, "far", far)
assert abs(near) < 0.5
assert far > 10 * max(abs(near), 0.01)

In [ ]:
# Example 3: Inception Score: uniform predictions give exactly 1; confident, varied predictions give the class count.
import torch
from torch_fidelity.metric_isc import isc_features_to_metric

uniform = torch.ones(1000, 10)
confident = torch.zeros(1000, 10)
confident[range(1000), [i % 10 for i in range(1000)]] = 100.0
u = isc_features_to_metric(uniform, splits=2, shuffle=False)["inception_score_mean"]
c = isc_features_to_metric(confident, splits=2, shuffle=False)["inception_score_mean"]
print("uniform", u, "confident", c)
assert abs(u - 1.0) < 1e-6
assert abs(c - 10.0) < 1e-3

In [ ]:
# Example 4: the Inception network, loaded from a weights FILE (no download), returns every feature layer.
import tempfile, torch
from unittest import mock
from torch_fidelity.feature_extractor_inceptionv3 import FeatureExtractorInceptionV3 as FE

# Our notebook must not download: save the freshly-initialised network as a weights file and load THAT file.
with mock.patch("torch_fidelity.feature_extractor_inceptionv3.load_state_dict_from_url", return_value={}), mock.patch.object(FE, "load_state_dict", lambda self, sd: None):
    state = FE("inception-v3-compat", ["64"]).state_dict()
path = tempfile.mkdtemp() + "/inception.pth"
torch.save(state, path)
net = FE("inception-v3-compat", ["64", "192", "2048", "logits_unbiased"], feature_extractor_weights_path=path)
x = (torch.rand(2, 3, 32, 32) * 255).to(torch.uint8)
out = net(x)
print([o.shape for o in out])
assert [tuple(o.shape) for o in out] == [(2, 64), (2, 192), (2, 2048), (2, 1008)]

In [ ]:
# Example 5: the full pipeline: FID and KID of two small in-memory datasets, from the weights file, on the CPU.
import tempfile, torch
from unittest import mock
from torch_fidelity import calculate_metrics
from torch_fidelity.feature_extractor_inceptionv3 import FeatureExtractorInceptionV3 as FE

with mock.patch("torch_fidelity.feature_extractor_inceptionv3.load_state_dict_from_url", return_value={}), mock.patch.object(FE, "load_state_dict", lambda self, sd: None):
    state = FE("inception-v3-compat", ["64"]).state_dict()
path = tempfile.mkdtemp() + "/inception.pth"
torch.save(state, path)
torch.manual_seed(0)
imgs = (torch.rand(32, 3, 32, 32) * 255).to(torch.uint8)
other = (torch.rand(32, 3, 32, 32) * 64).to(torch.uint8)
class Dataset(torch.utils.data.Dataset):
    def __init__(self, x): self.x = x
    def __len__(self): return len(self.x)
    def __getitem__(self, i): return self.x[i]
r = calculate_metrics(input1=Dataset(imgs), input2=Dataset(other), fid=True, kid=True, kid_subsets=2, kid_subset_size=16, cuda=False, verbose=False, cache=False, batch_size=8, save_cpu_ram=True, feature_extractor_weights_path=path)
print({k: round(float(v), 4) for k, v in r.items()})
assert set(r) == {"frechet_inception_distance", "kernel_inception_distance_mean", "kernel_inception_distance_std"}
assert r["frechet_inception_distance"] > 0